In [1]:
# First splitting the sentence
tokenized_text = "I've been waiting for this course my whole life".split()
print(tokenized_text)

["I've", 'been', 'waiting', 'for', 'this', 'course', 'my', 'whole', 'life']


# Here we are splitting word by word by a better approach to split the text by subtext. It has a benefit that no. of vocabulary decreases

In [3]:
# importing the pretrained transformer
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-cased") # if we know what type of tokenizer we want we can explicitly import that here Bert Tokenizer

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased') # If we don't know the tokenizer the autotokenizer will automatically detect


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [9]:
for key,values in tokenizer(tokenized_text).items():
  print(key, values)

input_ids [[101, 146, 112, 1396, 102], [101, 1151, 102], [101, 2613, 102], [101, 1111, 102], [101, 1142, 102], [101, 1736, 102], [101, 1139, 102], [101, 2006, 102], [101, 1297, 102]]
token_type_ids [[0, 0, 0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
attention_mask [[1, 1, 1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1], [1, 1, 1]]


### Here we got three values:
input_ids: convert text to integer and add CLS and SEP

token_type_ids:

attention_mask: It is added when we padding and want transformer to ignore it.

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "I've been waiting for this course my whole life"

tokens = tokenizer.tokenize(sequence)

print(tokens)

['I', "'", 've', 'been', 'waiting', 'for', 'this', 'course', 'my', 'whole', 'life']


In [12]:
# doing the above process manually
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[146, 112, 1396, 1151, 2613, 1111, 1142, 1736, 1139, 2006, 1297]


In [13]:
# from int to text
print(tokenizer.decode(ids))

I ' ve been waiting for this course my whole life


### Models expect a batch of input not one.

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for this course my whole life"

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)

model(input_ids)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (11) must match the size of tensor b (512) at non-singleton dimension 1

### Error is is because tokenizer function automatically add an additional dimension. Let's add additional dim

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for this course my whole life"

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])  # Adding dim here
print("Input Ids:", input_ids)

output = model(input_ids)

print("Logits:", output.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input Ids: tensor([[1045, 1005, 2310, 2042, 3403, 2005, 2023, 2607, 2026, 2878, 2166]])
Logits: tensor([[-3.4211,  3.5901]], grad_fn=<AddmmBackward0>)


In [ ]:
# Suppose the batched_ids is token of word. Both the sequence are not equal that why we add padding to make them equal.

padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


### The logits of first and second should be same because the values are same. What is happening is that padding is change that sequence which we don't want so we simply ignore it using attension mask.

In [19]:

batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1,1,1],
    [1,1,0]
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
